# Store Vision — fine-tune YOLOX-S on your inventory (box + packet)

Free-GPU fine-tune of `yolox-s` on your labeled frames, then re-export to ONNX.
This trains **only** box + packet (the two-model setup); person + vehicles keep
coming from the existing COCO model.

**Before you start:** Runtime → Change runtime type → **GPU** (T4 is free).

You need `dataset.zip` — produced locally by `finetune/scripts/prepare_dataset.py`
(zip the `finetune/dataset` folder so it contains `dataset/train2017`, 
`dataset/val2017`, `dataset/annotations`).


## 0. Confirm the GPU


In [ ]:
!nvidia-smi


## 1. Clone YOLOX + install deps (no onnx-simplifier)

We install only what's needed and use Colab's preinstalled CUDA torch.


In [ ]:
%cd /content
!git clone -q https://github.com/Megvii-BaseDetection/YOLOX.git
%cd /content/YOLOX
# extras only — do NOT reinstall torch (keep Colab's CUDA build)
!pip install -q onnx onnxruntime loguru thop tabulate pycocotools psutil 'numpy<2'
# YOLOX itself, no deps, no build isolation so it sees Colab's torch
!pip install -q --no-deps --no-build-isolation -e .


## 2. Pretrained weights (start point for the fine-tune)


In [ ]:
!wget -q https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_s.pth
!ls -la yolox_s.pth


## 3. Upload your dataset

Run the cell, pick your `dataset.zip`. It unzips to `/content/YOLOX/finetune_data/`.


In [ ]:
from google.colab import files
import os
up = files.upload()            # choose dataset.zip
zip_name = list(up.keys())[0]
!rm -rf /content/YOLOX/finetune_data && mkdir -p /content/YOLOX/finetune_data
!unzip -q "$zip_name" -d /content/YOLOX/finetune_data
!find /content/YOLOX/finetune_data -maxdepth 2 -type d


Set the dataset path below to the folder that directly contains `train2017/`,
`val2017/`, and `annotations/` (adjust if your zip nested it differently).


In [ ]:
import os
DATA_DIR = '/content/YOLOX/finetune_data/dataset'
assert os.path.isdir(os.path.join(DATA_DIR,'annotations')), 'fix DATA_DIR to point at the folder with annotations/'
os.environ['YOLOX_DATA_DIR'] = DATA_DIR
print('dataset:', DATA_DIR)


## 4. Write the custom experiment (2 classes: box, packet)


In [ ]:
%%writefile /content/YOLOX/exps/yolox_s_box_packet.py
import os
from yolox.exp import Exp as MyExp

class Exp(MyExp):
    def __init__(self):
        super().__init__()
        self.depth = 0.33
        self.width = 0.50
        self.exp_name = 'yolox_s_box_packet'
        self.num_classes = 2  # box, packet
        self.data_dir = os.environ.get('YOLOX_DATA_DIR', 'finetune_data/dataset')
        self.train_ann = 'instances_train2017.json'
        self.val_ann = 'instances_val2017.json'
        self.max_epoch = 50
        self.warmup_epochs = 1
        self.no_aug_epochs = 15
        self.eval_interval = 5
        self.print_interval = 20
        self.data_num_workers = 2
        self.mosaic_prob = 0.5
        self.mixup_prob = 0.5
        self.hsv_prob = 1.0
        self.flip_prob = 0.5
        self.input_size = (640, 640)
        self.test_size = (640, 640)
        self.basic_lr_per_img = 0.01 / 64.0


## 5. Train

Starts from `yolox_s.pth`. YOLOX will report a head-shape mismatch (80 -> 2
classes) and reinitialize the detection head — that is expected and correct.
~50 epochs on ~250 images is roughly 20-40 min on a T4. Lower `--max-epoch`
via the exp if you want a quicker first pass.


In [ ]:
!python tools/train.py -f exps/yolox_s_box_packet.py -d 1 -b 8 --fp16 -o -c yolox_s.pth


## 6. Export the fine-tuned model to ONNX

Robust across torch versions (handles the newer dynamo exporter).


In [ ]:
import torch
from yolox.exp import get_exp

CKPT = 'YOLOX_outputs/yolox_s_box_packet/best_ckpt.pth'
exp = get_exp('exps/yolox_s_box_packet.py', None)
model = exp.get_model()
ckpt = torch.load(CKPT, map_location='cpu')
model.load_state_dict(ckpt['model'])
model.eval()
model.head.decode_in_inference = False   # raw outputs; decode happens in the ONNX demo/runtime
dummy = torch.randn(1, 3, 640, 640)
kw = dict(input_names=['images'], output_names=['output'], opset_version=11)
try:
    torch.onnx.export(model, dummy, 'yolox_s_finetuned.onnx', dynamo=False, **kw)
except TypeError:
    torch.onnx.export(model, dummy, 'yolox_s_finetuned.onnx', **kw)
print('exported yolox_s_finetuned.onnx')


## 7. Download the model (and checkpoint)


In [ ]:
from google.colab import files
files.download('yolox_s_finetuned.onnx')
# optional: keep the torch checkpoint too
# files.download('YOLOX_outputs/yolox_s_box_packet/best_ckpt.pth')


## Next

Put `yolox_s_finetuned.onnx` into the repo's `models/` folder and tell me.
I'll wire up the **two-model runtime** (COCO model for person/vehicle +
this model for box/packet) and re-run it on your clips so you can see the
inventory it now catches.

If box/packet accuracy is weak, the usual fix is **more/varied labels** for the
weak class, then re-run this notebook.
